In [1]:
from shutil import rmtree

from langchain_ollama import ChatOllama, OllamaEmbeddings
from transformers import AutoTokenizer

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
CHROMA_STORAGE_PATH = "../data/vectors/chroma"
rmtree(CHROMA_STORAGE_PATH, ignore_errors=True)

In [3]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:4B",
    temperature=0,
    verbose=True,
    extract_reasoning=True,
)
embedding = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")

In [4]:
pdf_loaders = [
    PyPDFLoader("../data/pdf/cs229_lectures/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("../data/pdf/cs229_lectures/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("../data/pdf/cs229_lectures/MachineLearning-Lecture02.pdf"),
    PyPDFLoader("../data/pdf/cs229_lectures/MachineLearning-Lecture03.pdf"),
]

docs = [doc for loader in pdf_loaders for doc in loader.load()]

print(len(docs))
docs[:3]

78


[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2008-07-11T11:25:23-07:00', 'author': '', 'moddate': '2008-07-11T11:25:23-07:00', 'title': '', 'source': '../data/pdf/cs229_lectures/MachineLearning-Lecture01.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1'}, page_content='MachineLearning-Lecture01  \nInstructor (Andrew Ng): Okay. Good morning. Welcome to CS229, the machine \nlearning class. So what I wanna do today is just spend a little time going over the logistics \nof the class, and then we\'ll start to talk a bit about machine learning.  \nBy way of introduction, my name\'s Andrew Ng and I\'ll be instructor for this class. And so \nI personally work in machine learning, and I\'ve worked on it for about 15 years now, and \nI actually think that machine learning is the most exciting field of all the computer \nsciences. So I\'m actually always excited about teaching this class. Sometimes I actually \n

In [5]:
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=256,
    chunk_overlap=32,
)
splits = text_splitter.split_documents(docs)
print(len(splits))
splits[:5]

278


[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2008-07-11T11:25:23-07:00', 'author': '', 'moddate': '2008-07-11T11:25:23-07:00', 'title': '', 'source': '../data/pdf/cs229_lectures/MachineLearning-Lecture01.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1'}, page_content="MachineLearning-Lecture01  \nInstructor (Andrew Ng): Okay. Good morning. Welcome to CS229, the machine \nlearning class. So what I wanna do today is just spend a little time going over the logistics \nof the class, and then we'll start to talk a bit about machine learning.  \nBy way of introduction, my name's Andrew Ng and I'll be instructor for this class. And so \nI personally work in machine learning, and I've worked on it for about 15 years now, and \nI actually think that machine learning is the most exciting field of all the computer \nsciences. So I'm actually always excited about teaching this class. Sometimes I actually \nthink

In [6]:
vector_db = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=CHROMA_STORAGE_PATH,
)
vector_db._collection.count()

278

In [7]:
result = vector_db.similarity_search("is there an email I can ask for help", k=3)

print(result[0].page_content, "\n==============")
print(result[1].page_content, "\n==============")
print(result[2].page_content, "\n==============")

newsgroup that's sort of a forum for people in the class to get to know each other and 
have whatever discussions you want to have amongst yourselves. So the class newsgroup 
will not be monitored by the TAs and me. But this is a place for you to form study groups 
or find project partners or discuss homework problems and so on, and it's not monitored 
by the TAs and me. So feel free to talk trash about this class there.  
If you want to contact the teaching staff, please use the email address written down here, 
cs229-qa@cs.stanford.edu. This goes to an account that's read by all the TAs and me. So 
rather than sending us email individually, if you send email to this account, it will 
actually let us get back to you maximally quickly with answers to your questions.  
If you're asking questions about homework problems, please say in the subject line which 
assignment and which question the email refers to, since that will also help us to route 
your question to the appropriate TA or to

In [8]:
result = vector_db.similarity_search(
    query="what did they say about regression in the third lecture?",
    filter={
        "source": "../data/pdf/cs229_lectures/MachineLearning-Lecture03.pdf",
    },
    k=3,
)
print("\n==============\n", result[0].page_content)
print("\n==============\n", result[1].page_content)
print("\n==============\n", result[2].page_content)


 MachineLearning-Lecture03  
Instructor (Andrew Ng):Okay. Good morning and welcome back to the third lecture of 
this class. So here’s what I want to do today, and some of the topics I do today may seem 
a little bit like I’m jumping, sort of, from topic to topic, but here’s, sort of, the outline for 
today and the illogical flow of ideas. In the last lecture, we talked about linear regression 
and today I want to talk about sort of an adaptation of that called locally weighted 
regression. It’s very a popular algorithm that’s actually one of my former mentors 
probably favorite machine learning algorithm.  
We’ll then talk about a probable second interpretation of linear regression and use that to 
move onto our first classification algorithm, which is logistic regression; take a brief 
digression to tell you about something called the perceptron algorithm, which is 
something we’ll come back to, again, later this quarter; and time allowing I hope to get to 
Newton’s method, which is